In [ ]:
%pip -q install numpy pandas matplotlib

In [ ]:
from google.colab import files
from pathlib import Path
from zipfile import ZipFile

uploaded_zip = files.upload()
zip_name = next(name for name in uploaded_zip if name.lower().endswith(".zip"))

data_dir = Path("historical_data")
data_dir.mkdir(exist_ok=True)

with ZipFile(zip_name) as archive:
    assert archive.testzip() is None, "The ZIP failed its integrity check."
    archive.extractall(data_dir)

print("Extracted:", zip_name)

In [ ]:
import json
import numpy as np
from pathlib import Path
from google.colab import files

required_sites = ["site01", "site04", "site07", "site11"]
horizons = {}

for site_id in required_sites:
    print(f"Upload the horizon JSON for {site_id}")
    uploaded_site = files.upload()
    filename = next(name for name in uploaded_site if name.lower().endswith(".json"))
    site = json.loads(Path(filename).read_text(encoding="utf-8"))

    assert site["schema_version"] == 2
    assert site["site_id"] == site_id
    assert site["frame"] == "MOON_ME_DE421"
    assert site["coverage_status"] == "complete"
    horizons[site_id] = site

ephemerides = {}

for site_id in required_sites:
    stem = f"ephemeris_{site_id}_2009-07_to_2013-07"
    npz_path = Path("historical_data") / f"{stem}.npz"
    json_path = Path("historical_data") / f"{stem}.json"

    metadata = json.loads(json_path.read_text(encoding="utf-8"))
    assert metadata["site_id"] == site_id
    assert metadata["frame"] == "MOON_ME_DE421"
    assert metadata["abcorr"] == "CN+S"
    assert metadata["spkcpo_refloc"] == "OBSERVER"
    assert np.allclose(
        metadata["observer_position_xyz_m"],
        horizons[site_id]["observer_xyz_m"],
    )

    with np.load(npz_path) as data:
        assert data["times_s"].shape == (420_769,)
        assert data["sun_vectors_m"].shape == (420_769, 3)
        assert data["earth_vectors_m"].shape == (420_769, 3)
        assert np.all(np.diff(data["times_s"]) > 0)
        assert np.isfinite(data["sun_vectors_m"]).all()
        assert np.isfinite(data["earth_vectors_m"]).all()

        ephemerides[site_id] = {
            "times_s": data["times_s"].copy(),
            "sun_vectors_m": data["sun_vectors_m"].copy(),
            "earth_vectors_m": data["earth_vectors_m"].copy(),
            "metadata": metadata,
        }

    print(site_id, "verified:", len(ephemerides[site_id]["times_s"]), "samples")

print("All four sites are ready for the analysis step.")

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

def azimuth_elevation(vectors_m, latitude_deg, longitude_deg):
    lat, lon = np.deg2rad([latitude_deg, longitude_deg])

    east = np.array([-np.sin(lon), np.cos(lon), 0.0])
    north = np.array([
        -np.sin(lat) * np.cos(lon),
        -np.sin(lat) * np.sin(lon),
        np.cos(lat),
    ])
    up = np.array([
        np.cos(lat) * np.cos(lon),
        np.cos(lat) * np.sin(lon),
        np.sin(lat),
    ])

    e = vectors_m @ east
    n = vectors_m @ north
    u = vectors_m @ up

    az = np.mod(np.rad2deg(np.arctan2(e, n)), 360.0)
    el = np.rad2deg(np.arctan2(u, np.hypot(e, n)))
    return az, el

def horizon_at(azimuth_deg, horizon_az, horizon_el):
    return np.interp(
        np.mod(azimuth_deg, 360.0),
        np.append(horizon_az, 360.0),
        np.append(horizon_el, horizon_el[0]),
    )

site_ids = ["site01", "site04", "site07", "site11"]
times_s = ephemerides["site01"]["times_s"]
n_days = int(round(times_s[-1] / 86400.0))
day_index = np.minimum(
    (times_s // 86400.0).astype(int),
    n_days - 1,
)
day_counts = np.bincount(day_index, minlength=n_days)

summary_rows = []
daily_rows = []

for site_id in site_ids:
    eph = ephemerides[site_id]
    horizon = horizons[site_id]

    assert np.array_equal(eph["times_s"], times_s)

    sun_az, sun_el = azimuth_elevation(
        eph["sun_vectors_m"],
        float(horizon["lat_deg"]),
        float(horizon["lon_deg"]),
    )
    earth_az, earth_el = azimuth_elevation(
        eph["earth_vectors_m"],
        float(horizon["lat_deg"]),
        float(horizon["lon_deg"]),
    )

    horizon_az = np.asarray(horizon["az_deg"], dtype=float)
    horizon_el = np.asarray(horizon["horizon_deg"], dtype=float)

    sun_clearance_deg = sun_el - horizon_at(
        sun_az, horizon_az, horizon_el
    )
    earth_clearance_deg = earth_el - horizon_at(
        earth_az, horizon_az, horizon_el
    )

    sun_center_visible = sun_clearance_deg >= 0.0
    earth_center_visible = earth_clearance_deg >= 0.0

    summary_rows.append({
        "site_id": site_id,
        "samples": len(times_s),
        "sun_center_above_horizon_pct": 100.0 * sun_center_visible.mean(),
        "earth_center_above_horizon_pct": 100.0 * earth_center_visible.mean(),
    })

    daily_rows.append(pd.DataFrame({
        "day_from_start": np.arange(n_days),
        "site_id": site_id,
        "sun_center_above_horizon_fraction":
            np.bincount(day_index, weights=sun_center_visible, minlength=n_days)
            / day_counts,
        "earth_center_above_horizon_fraction":
            np.bincount(day_index, weights=earth_center_visible, minlength=n_days)
            / day_counts,
    }))

summary_df = pd.DataFrame(summary_rows)
daily_df = pd.concat(daily_rows, ignore_index=True)

display(summary_df.round(2))

fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

for site_id in site_ids:
    site_daily = daily_df[daily_df["site_id"] == site_id]
    axes[0].plot(
        site_daily["day_from_start"],
        site_daily["sun_center_above_horizon_fraction"],
        label=site_id,
    )
    axes[1].plot(
        site_daily["day_from_start"],
        site_daily["earth_center_above_horizon_fraction"],
        label=site_id,
    )

axes[0].set_title("Sun centre above terrain horizon")
axes[1].set_title("Earth centre above terrain horizon")
axes[0].set_ylabel("Fraction of samples")
axes[1].set_ylabel("Fraction of samples")
axes[1].set_xlabel("Elapsed days from 2009-07-01")
axes[0].set_ylim(0, 1)
axes[1].set_ylim(0, 1)

for ax in axes:
    ax.grid(True, alpha=0.3)
    ax.legend()

plt.tight_layout()
plt.show()

summary_df.to_csv("four_year_center_visibility_summary.csv", index=False)
daily_df.to_csv("four_year_daily_center_visibility.csv", index=False)
print("Saved the full-period summary and daily trend CSVs.")

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path

# 256 evenly spaced samples across the apparent solar disc.
# Increase this for a more precise estimate at the cost of runtime.
N_DISC_POINTS = 256
SUN_RADIUS_M = 695_700_000.0
EARTH_RADIUS_M = 6_378_136.6
SUN_THRESHOLD = 0.80
CHUNK_SIZE = 4000

# Uniform points in a circular disc (Fibonacci spiral).
k = np.arange(N_DISC_POINTS, dtype=float)
r_disc = np.sqrt((k + 0.5) / N_DISC_POINTS)
theta_disc = k * (np.pi * (3.0 - np.sqrt(5.0)))
disk_x = r_disc * np.cos(theta_disc)
disk_y = r_disc * np.sin(theta_disc)

def unit_rows(v):
    return v / np.linalg.norm(v, axis=1, keepdims=True)

summary_rows = []
daily_frames = []

for site_id in ["site01", "site04", "site07", "site11"]:
    eph = ephemerides[site_id]
    horizon_doc = horizons[site_id]

    # Terrain horizon profile and missing-data mask.
    az_profile = np.asarray(horizon_doc["az_deg"], dtype=float)
    horizon_profile = np.asarray(horizon_doc["horizon_deg"], dtype=float)
    valid_profile = np.asarray(horizon_doc["valid"], dtype=bool)

    if not valid_profile.all():
        raise ValueError(f"{site_id} has invalid horizon directions; cannot run this estimate.")

    # Local north/east/up basis from the site’s planetocentric latitude/longitude.
    lat = np.deg2rad(float(horizon_doc["lat_deg"]))
    lon = np.deg2rad(float(horizon_doc["lon_deg"]))

    up = np.array([np.cos(lat) * np.cos(lon),
                   np.cos(lat) * np.sin(lon),
                   np.sin(lat)])
    east = np.array([-np.sin(lon), np.cos(lon), 0.0])
    north = np.array([-np.sin(lat) * np.cos(lon),
                      -np.sin(lat) * np.sin(lon),
                       np.cos(lat)])

    sun = np.asarray(eph["sun_vectors_m"], dtype=float)
    earth = np.asarray(eph["earth_vectors_m"], dtype=float)
    times_s = np.asarray(eph["times_s"], dtype=float)

    n = len(times_s)
    sun_fraction = np.empty(n, dtype=np.float32)
    earth_center_visible = np.empty(n, dtype=bool)

    for start in range(0, n, CHUNK_SIZE):
        stop = min(start + CHUNK_SIZE, n)

        s_vec = sun[start:stop]
        e_vec = earth[start:stop]
        s_hat = unit_rows(s_vec)
        e_hat = unit_rows(e_vec)

        # Earth centre visibility against terrain.
        e_up = e_hat @ up
        e_north = e_hat @ north
        e_east = e_hat @ east
        e_az = np.mod(np.rad2deg(np.arctan2(e_east, e_north)), 360.0)
        e_el = np.rad2deg(np.arcsin(np.clip(e_up, -1.0, 1.0)))

        e_idx = e_az / (az_profile[1] - az_profile[0])
        e_i0 = np.floor(e_idx).astype(int) % len(horizon_profile)
        e_i1 = (e_i0 + 1) % len(horizon_profile)
        e_frac = e_idx - np.floor(e_idx)
        e_horizon = horizon_profile[e_i0] * (1.0 - e_frac) + horizon_profile[e_i1] * e_frac
        earth_center_visible[start:stop] = e_el > e_horizon

        # Build two perpendicular directions spanning the Sun's apparent disc.
        # Project local east into the plane perpendicular to the Sun direction.
        b1 = east[None, :] - (s_hat @ east)[:, None] * s_hat
        b1_norm = np.linalg.norm(b1, axis=1)
        bad = b1_norm < 1e-12
        if np.any(bad):
            fallback = north[None, :] - (s_hat[bad] @ north)[:, None] * s_hat[bad]
            b1[bad] = fallback
            b1_norm[bad] = np.linalg.norm(fallback, axis=1)
        b1 /= b1_norm[:, None]
        b2 = np.cross(s_hat, b1)

        # Angular radii of Sun and Earth as seen from the site.
        sun_alpha = np.arcsin(np.clip(SUN_RADIUS_M / np.linalg.norm(s_vec, axis=1), 0, 1))
        earth_alpha = np.arcsin(np.clip(EARTH_RADIUS_M / np.linalg.norm(e_vec, axis=1), 0, 1))
        earth_cos_radius = np.cos(earth_alpha)

        visible_count = np.zeros(stop - start, dtype=np.int32)

        # Test each sampled point on the Sun disc against terrain and Earth.
        for dx, dy in zip(disk_x, disk_y):
            ray = (
                s_hat
                + (np.tan(sun_alpha) * dx)[:, None] * b1
                + (np.tan(sun_alpha) * dy)[:, None] * b2
            )
            ray = unit_rows(ray)

            ray_up = ray @ up
            ray_north = ray @ north
            ray_east = ray @ east
            ray_az = np.mod(np.rad2deg(np.arctan2(ray_east, ray_north)), 360.0)
            ray_el = np.rad2deg(np.arcsin(np.clip(ray_up, -1.0, 1.0)))

            ray_idx = ray_az / (az_profile[1] - az_profile[0])
            i0 = np.floor(ray_idx).astype(int) % len(horizon_profile)
            i1 = (i0 + 1) % len(horizon_profile)
            frac = ray_idx - np.floor(ray_idx)
            terrain_horizon = horizon_profile[i0] * (1.0 - frac) + horizon_profile[i1] * frac

            clear_of_terrain = ray_el > terrain_horizon
            clear_of_earth = np.einsum("ij,ij->i", ray, e_hat) < earth_cos_radius
            visible_count += (clear_of_terrain & clear_of_earth)

        sun_fraction[start:stop] = visible_count / N_DISC_POINTS

    summary_rows.append({
        "site_id": site_id,
        "samples": n,
        "mean_sun_visible_fraction_pct": 100.0 * float(np.mean(sun_fraction)),
        "samples_with_sun_fraction_ge_80_pct": 100.0 * float(np.mean(sun_fraction >= SUN_THRESHOLD)),
        "earth_center_above_terrain_pct": 100.0 * float(np.mean(earth_center_visible)),
        "sun_fraction_min": float(np.min(sun_fraction)),
        "sun_fraction_max": float(np.max(sun_fraction)),
    })

    # Daily averages for plotting and sharing.
    start_utc = pd.to_datetime(eph["metadata"]["start_utc"], utc=True)
    utc_times = start_utc + pd.to_timedelta(times_s, unit="s")
    daily = pd.DataFrame({
        "utc": utc_times,
        "sun_visible_fraction": sun_fraction,
        "earth_center_visible": earth_center_visible,
    }).set_index("utc").resample("1D").mean()

    daily["site_id"] = site_id
    daily_frames.append(daily.reset_index())

    print(f"{site_id}: completed {n:,} samples")

summary_df = pd.DataFrame(summary_rows)
daily_df = pd.concat(daily_frames, ignore_index=True)

display(summary_df)

summary_df.to_csv("four_year_visibility_summary_estimate.csv", index=False)
daily_df.to_csv("four_year_daily_visibility_estimate.csv", index=False)

print("Saved four_year_visibility_summary_estimate.csv")
print("Saved four_year_daily_visibility_estimate.csv")

In [ ]:
from zipfile import ZipFile, ZIP_DEFLATED
from pathlib import Path
from google.colab import files

result_files = [
    Path("four_year_visibility_summary_estimate.csv"),
    Path("four_year_daily_visibility_estimate.csv"),
    Path("four_year_sun_any_part_sample_level.csv"),
]

for path in result_files:
    assert path.exists(), f"Missing {path.name}. Run the calculation cells first."

zip_name = "four_year_visibility_results.zip"

with ZipFile(zip_name, "w", ZIP_DEFLATED) as archive:
    for path in result_files:
        archive.write(path, arcname=path.name)

files.download(zip_name)

In [ ]:
import numpy as np
import pandas as pd

# Same 256-point solar-disc sampling used in the earlier estimate.
N_DISC_POINTS = 256
SUN_RADIUS_M = 695_700_000.0
EARTH_RADIUS_M = 6_378_136.6
CHUNK_SIZE = 4000

k = np.arange(N_DISC_POINTS, dtype=float)
r_disc = np.sqrt((k + 0.5) / N_DISC_POINTS)
theta_disc = k * (np.pi * (3.0 - np.sqrt(5.0)))
disk_x = r_disc * np.cos(theta_disc)
disk_y = r_disc * np.sin(theta_disc)

def unit_rows(v):
    return v / np.linalg.norm(v, axis=1, keepdims=True)

results = []

for site_id in ["site01", "site04", "site07", "site11"]:
    eph = ephemerides[site_id]
    horizon_doc = horizons[site_id]

    az_profile = np.asarray(horizon_doc["az_deg"], dtype=float)
    horizon_profile = np.asarray(horizon_doc["horizon_deg"], dtype=float)
    valid = np.asarray(horizon_doc["valid"], dtype=bool)

    if not valid.all():
        raise ValueError(f"{site_id} has invalid horizon directions.")

    az_step = az_profile[1] - az_profile[0]
    if not np.allclose(np.diff(az_profile), az_step):
        raise ValueError(f"{site_id} azimuth samples are not evenly spaced.")

    lat = np.deg2rad(float(horizon_doc["lat_deg"]))
    lon = np.deg2rad(float(horizon_doc["lon_deg"]))

    up = np.array([
        np.cos(lat) * np.cos(lon),
        np.cos(lat) * np.sin(lon),
        np.sin(lat),
    ])
    east = np.array([-np.sin(lon), np.cos(lon), 0.0])
    north = np.array([
        -np.sin(lat) * np.cos(lon),
        -np.sin(lat) * np.sin(lon),
        np.cos(lat),
    ])

    sun = np.asarray(eph["sun_vectors_m"], dtype=float)
    earth = np.asarray(eph["earth_vectors_m"], dtype=float)
    n_samples = len(sun)

    # Count sampled Sun-disc points visible above terrain,
    # and separately count points not occulted by Earth.
    terrain_counts = np.zeros(n_samples, dtype=np.uint16)
    terrain_and_earth_counts = np.zeros(n_samples, dtype=np.uint16)

    for start in range(0, n_samples, CHUNK_SIZE):
        stop = min(start + CHUNK_SIZE, n_samples)

        s_vec = sun[start:stop]
        e_vec = earth[start:stop]
        s_hat = unit_rows(s_vec)
        e_hat = unit_rows(e_vec)

        b1 = east[None, :] - (s_hat @ east)[:, None] * s_hat
        b1_norm = np.linalg.norm(b1, axis=1)
        bad = b1_norm < 1e-12

        if np.any(bad):
            fallback = north[None, :] - (s_hat[bad] @ north)[:, None] * s_hat[bad]
            b1[bad] = fallback
            b1_norm[bad] = np.linalg.norm(fallback, axis=1)

        b1 /= b1_norm[:, None]
        b2 = np.cross(s_hat, b1)

        sun_alpha = np.arcsin(
            np.clip(SUN_RADIUS_M / np.linalg.norm(s_vec, axis=1), 0, 1)
        )
        earth_alpha = np.arcsin(
            np.clip(EARTH_RADIUS_M / np.linalg.norm(e_vec, axis=1), 0, 1)
        )
        earth_cos_radius = np.cos(earth_alpha)

        terrain_chunk = np.zeros(stop - start, dtype=np.uint16)
        both_chunk = np.zeros(stop - start, dtype=np.uint16)

        for dx, dy in zip(disk_x, disk_y):
            ray = (
                s_hat
                + (np.tan(sun_alpha) * dx)[:, None] * b1
                + (np.tan(sun_alpha) * dy)[:, None] * b2
            )
            ray = unit_rows(ray)

            ray_up = ray @ up
            ray_north = ray @ north
            ray_east = ray @ east

            ray_az = np.mod(
                np.rad2deg(np.arctan2(ray_east, ray_north)), 360.0
            )
            ray_el = np.rad2deg(
                np.arcsin(np.clip(ray_up, -1.0, 1.0))
            )

            idx = ray_az / az_step
            i0 = np.floor(idx).astype(int) % len(horizon_profile)
            i1 = (i0 + 1) % len(horizon_profile)
            frac = idx - np.floor(idx)
            terrain_horizon = (
                horizon_profile[i0] * (1.0 - frac)
                + horizon_profile[i1] * frac
            )

            clear_terrain = ray_el > terrain_horizon
            clear_earth = (
                np.einsum("ij,ij->i", ray, e_hat) < earth_cos_radius
            )

            terrain_chunk += clear_terrain
            both_chunk += clear_terrain & clear_earth

        terrain_counts[start:stop] = terrain_chunk
        terrain_and_earth_counts[start:stop] = both_chunk

    # NASA-map-style comparison: any sampled part of Sun visible over terrain.
    # The second value also includes Earth blocking the Sun.
    results.append({
        "site_id": site_id,
        "samples": n_samples,
        "sun_any_part_visible_terrain_only_pct":
            100.0 * np.mean(terrain_counts > 0),
        "sun_any_part_visible_including_earth_occultation_pct":
            100.0 * np.mean(terrain_and_earth_counts > 0),
    })

    print(f"{site_id}: completed {n_samples:,} samples")

any_part_df = pd.DataFrame(results)
display(any_part_df.round(3))
any_part_df.to_csv("four_year_sun_any_part_sample_level.csv", index=False)
print("Saved four_year_sun_any_part_sample_level.csv")

In [ ]:
Path("four_year_sun_any_part_sample_level.csv"),Path("four_year_sun_any_part_sample_level.csv"),